In [0]:
from pyspark.sql.functions import col, last, first, when, lit, avg, sum as _sum
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, LongType, FloatType

bronze_wdi_df = spark.read.table("workspace.default.bronze_wdi_df")
bronze_growup_df = spark.read.table("workspace.default.bronze_growup_df")
bronze_vdem_df = spark.read.table("workspace.default.bronze_vdem_df")

In [0]:
silver_wdi_df = bronze_wdi_df.drop('Country Code', 'Indicator Code')
silver_wdi_df.createOrReplaceTempView("wdi_raw")

# Column cleaning - casting years as integers
columns = silver_wdi_df.columns
year_columns = [col for col in columns[2:] if col.replace('.', '').isdigit()]

year_cast_sql = ", ".join([f"TRY_CAST(`{year}` AS DOUBLE) AS `{year}`" for year in year_columns])
year_cols_sql = ", ".join(year_columns)

# Assessing missing values using sql
year_missing_sql = " + ".join([f"CASE WHEN `{year}` IS NULL THEN 1 ELSE 0 END" for year in year_columns])

total_missing_sql = spark.sql(f"""
    SELECT 
        SUM(
            {year_missing_sql}
        ) as total_missing
    FROM wdi_raw
""")

missing_stats = spark.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(*) * {len(columns)} as total_cells,
        SUM({year_missing_sql}) as total_missing,
        ROUND(SUM({year_missing_sql}) / (COUNT(*) * {len(columns)}) * 100, 2) as missing_percentage
    FROM wdi_raw
""")

missing_stats.show()

In [0]:
# Indicator completeness - selecting indicators based on which ones have the most available data (pre-imputation)

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW indicator_completeness AS
    SELECT 
        `Indicator Name` as indicator_name,
        COUNT(*) as total_rows,
        AVG(
            {year_missing_sql}
        ) as avg_missing_count,
        MIN(
            {year_missing_sql}
        ) as min_missing_count,
        MAX(
            {year_missing_sql}
        ) as max_missing_count,
        COUNT(*) as count
    FROM wdi_raw
    GROUP BY `Indicator Name`
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW top_500_complete AS
    SELECT 
        indicator_name,
        ROUND(avg_missing_count, 2) as missing_count_mean,
        min_missing_count as missing_count_min,
        max_missing_count as missing_count_max
    FROM indicator_completeness
    ORDER BY avg_missing_count ASC
    LIMIT 500
""")

# Redundant population columns to remove
remove_pop = ['Population ages 0-14 (% of total population)', 'Population ages 0-14, female (% of female population)', 'Population ages 0-14, male (% of male population)', 'Population ages 00-04, female (% of female population)', 'Population ages 00-04, male (% of male population)', 'Population ages 05-09, female (% of female population)', 'Population ages 05-09, male (% of male population)', 'Population ages 10-14, female (% of female population)', 'Population ages 10-14, male (% of male population)', 'Population ages 15-19, female (% of female population)', 'Population ages 15-19, male (% of male population)', 'Population ages 15-64 (% of total population)', 'Population ages 15-64, female (% of female population)', 'Population ages 15-64, male (% of male population)', 'Population ages 20-24, female (% of female population)', 'Population ages 20-24, male (% of male population)', 'Population ages 25-29, female (% of female population)', 'Population ages 25-29, male (% of male population)', 'Population ages 30-34, female (% of female population)', 'Population ages 30-34, male (% of male population)', 'Population ages 35-39, female (% of female population)', 'Population ages 35-39, male (% of male population)', 'Population ages 40-44, female (% of female population)', 'Population ages 40-44, male (% of male population)', 'Population ages 45-49, female (% of female population)', 'Population ages 45-49, male (% of male population)', 'Population ages 50-54, female (% of female population)', 'Population ages 50-54, male (% of male population)', 'Population ages 55-59, female (% of female population)', 'Population ages 55-59, male (% of male population)', 'Population ages 60-64, female (% of female population)', 'Population ages 60-64, male (% of male population)', 'Population ages 65 and above (% of total population)', 'Population ages 65 and above, female (% of female population)', 'Population ages 65 and above, male (% of male population)', 'Population ages 65-69, female (% of female population)', 'Population ages 65-69, male (% of male population)', 'Population ages 70-74, female (% of female population)', 'Population ages 70-74, male (% of male population)', 'Population ages 75-79, female (% of female population)', 'Population ages 75-79, male (% of male population)', 'Population ages 80 and above, female (% of female population)', 'Population ages 80 and above, male (% of male population)', 'Population ages 0-14, female', 'Population ages 0-14, male','Population ages 0-14, total', 'Population ages 15-64, female','Population ages 15-64, male', 'Population ages 15-64, total','Population ages 65 and above, female','Population ages 65 and above, male', 'Population ages 65 and above, total']

pop_sql = ", ".join([f"'{item}'" for item in remove_pop])

top_250_indicators = spark.sql(f"""
          SELECT indicator_name
          FROM top_500_complete
          WHERE indicator_name NOT IN ({pop_sql})
          LIMIT 250 
          """)

top_250_indicators.createOrReplaceTempView("top_250_indicators")

top_250_list = [row.indicator_name for row in top_250_indicators.collect()]

# print(top_250_list)

indicator_list_str = ", ".join([f"'{ind.strip()}'" for ind in top_250_list])

filtered = spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW filtered AS
    SELECT *
    FROM wdi_raw
    WHERE `Indicator Name` IN ({indicator_list_str})
""")

top_250_df = spark.sql("SELECT indicator_name FROM top_250_indicators") 

filtered_df = (spark.table("wdi_raw")
    .join(top_250_df, 
          col("Indicator Name") == col("indicator_name"), 
          "inner").select(columns)
)

filtered_df.createOrReplaceTempView("filtered")
print(f"Count after filtering to top 250 indicators: {filtered_df.count()}")

filtered_df.show(5)


In [0]:
# Dropping rows with < 10 years of data 

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW wdi_filtered_rows AS
    SELECT *
    FROM filtered
    WHERE ({len(year_columns)} - ({year_missing_sql})) >= 10
""")


# Re-assess missing values after filtering
filtered_stats = spark.sql(f"""
    SELECT 
        COUNT(*) as row_count,
        COUNT(*) * {len(columns)} as total_cells,
        SUM({year_missing_sql}) as total_missing,
        ROUND(SUM({year_missing_sql}) / (COUNT(*) * {len(columns)}) * 100, 2) as missing_percentage
    FROM wdi_filtered_rows
""")

wdi_filtered_df = spark.table('wdi_filtered_rows')
filtered_stats.show()

In [0]:
# Missing value imputation - Forward filling missing values with most recent obs and subsequently backfilling any unfilled values

# Forward/back filling using Window function
unpivot_exprs = [f"stack({len(year_columns)}, {', '.join([f"'{y}', `{y}`" for y in year_columns])}) as (year, value)"]

unpivoted = wdi_filtered_df.select('Country Name', 'Indicator Name', *[col(y) for y in year_columns]) \
    .selectExpr('`Country Name`', '`Indicator Name`', *unpivot_exprs)

window_spec = Window.partitionBy('Country Name', 'Indicator Name').orderBy('year')

# Forward fill
filled = unpivoted.withColumn('value_ffill', last('value', ignorenulls=True).over(window_spec))

# Backward fill
window_spec_desc = Window.partitionBy('Country Name', 'Indicator Name').orderBy(col('year').desc())

filled = filled.withColumn('value_bfill', first('value_ffill', ignorenulls=True).over(window_spec_desc))

final_filled = filled.withColumn(
    'value',
    F.coalesce(F.col('value_ffill'), F.col('value_bfill'))
).drop('value_ffill', 'value_bfill')

# Pivot back and drop
silver_wdi_df = final_filled.groupBy('Country Name', 'Indicator Name').pivot('year').agg(first('value'))

print(f"WDI Imputed dataset shape: {silver_wdi_df.count()} rows x {len(silver_wdi_df.columns)} columns")

null_counts = silver_wdi_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in silver_wdi_df.columns
]).collect()[0]

total_missing = sum(null_counts.asDict().values())

print(f"WDI Missing values: {total_missing}")

silver_wdi_df.show()

# Saving WDI data to table
silver_wdi_df.write.option("delta.columnMapping.mode", "name").format("delta").mode("overwrite").saveAsTable("silver_wdi_df")


In [0]:
# Row Selection and cleaning of GROWup Dataset

silver_growup_df = bronze_growup_df.drop('Country Code', 'Indicator Code')

keep_cols = ["year","countryname","onset_ko_flag","onset_ko_terr_flag","onset_ko_gov_flag","onset_ko_eth_flag","onset_ko_noneth_flag","onset_ko_terr_eth_flag","onset_ko_gov_eth_flag","onset_ko_terr_noneth_flag","onset_ko_gov_noneth_flag","incidence_flag","incidence_terr_flag","incidence_gov_flag","incidence_eth_flag","incidence_noneth_flag","incidence_terr_eth_flag","incidence_gov_eth_flag","incidence_terr_noneth_flag","incidence_gov_noneth_flag","onset_do_flag","onset_do_terr_flag","onset_do_gov_flag","onset_do_eth_flag","onset_do_noneth_flag","onset_do_terr_eth_flag","onset_do_gov_eth_flag","onset_do_terr_noneth_flag","onset_do_gov_noneth_flag","warhist","warhist_terr","warhist_gov","peaceyears","peaceyears_terr","peaceyears_gov"]

binary_cols = ["onset_ko_flag","onset_ko_terr_flag","onset_ko_gov_flag","onset_ko_eth_flag","onset_ko_noneth_flag","onset_ko_terr_eth_flag","onset_ko_gov_eth_flag","onset_ko_terr_noneth_flag","onset_ko_gov_noneth_flag","incidence_flag","incidence_terr_flag","incidence_gov_flag","incidence_eth_flag","incidence_noneth_flag","incidence_terr_eth_flag","incidence_gov_eth_flag","incidence_terr_noneth_flag","incidence_gov_noneth_flag","onset_do_flag","onset_do_terr_flag","onset_do_gov_flag","onset_do_eth_flag","onset_do_noneth_flag","onset_do_terr_eth_flag","onset_do_gov_eth_flag","onset_do_terr_noneth_flag","onset_do_gov_noneth_flag"]
silver_growup_df = silver_growup_df[keep_cols]

# Imputing missing values for binary categories marking onset of new armed conflict with 0's 
silver_growup_df = silver_growup_df.na.fill(value=0, subset=binary_cols)

# Converting GROWup dataset to WDI-like time series format
silver_growup_df.createOrReplaceTempView("growup_raw")
metadata_cols = ['countryname', 'year']
indicator_cols = [c for c in silver_growup_df.columns if c not in metadata_cols]

# Unpivoting using stack
stack_expr = f"stack({len(indicator_cols)}, " + \
             ", ".join([f"'{ind}', `{ind}`" for ind in indicator_cols]) + \
             ") as (`Indicator Name`, value)"

gu_long = silver_growup_df.selectExpr(
    'countryname',
    'year',
    stack_expr
)

print(f"Melted dataframe shape: {gu_long.count()} rows x {len(gu_long.columns)} columns")
print("\nFirst few rows:")
gu_long.show(5)
print(f"\nUnique indicators: {gu_long.select('Indicator Name').distinct().count()}")

gu_long.createOrReplaceTempView("growup_long")

#Pivoting to get years as columns (WDI-like structure)
gu_like_wdi = gu_long.groupBy('countryname', 'Indicator Name') \
    .pivot('year') \
    .agg(first('value')) \

gu_like_wdi = gu_like_wdi.withColumnRenamed('countryname', 'Country Name') 

# Renaming columns to string type for consistency
for col_name in gu_like_wdi.columns:
    if col_name not in ['Country Name', 'Indicator Name']:
        gu_like_wdi = gu_like_wdi.withColumnRenamed(col_name, str(col_name))

print("\nReshaped dataframe - Structure like WDI:")
print(f"Shape: {gu_like_wdi.count()} rows x {len(gu_like_wdi.columns)} columns")
print("\nColumn names (first 10):")
print(gu_like_wdi.columns[:10])

gu_like_wdi.createOrReplaceTempView("growup_wdi_style")


In [0]:
gu_like_wdi.createOrReplaceTempView("growup_wdi_style")

# Filtering to keep only years spanned by WDI (1960-2020)
year_range = [str(year) for year in range(1960, 2021)]
keep_cols = ['Country Name', 'Indicator Name'] + [year for year in year_range if year in gu_like_wdi.columns]

gu_like_wdi_filtered = gu_like_wdi[keep_cols]

year_columns = [c for c in gu_like_wdi_filtered.columns 
                if c not in ['Country Name', 'Indicator Name']]

gu_like_wdi_filtered.createOrReplaceTempView("growup_filtered")

# Imputing remaining missing values (non-binary categories) with same forward/backward fill method as WDI (as not all indicator, country pairs have data for every year after converting years to cols)
unpivot_exprs = [f"stack({len(year_columns)}, {', '.join([f"'{y}', `{y}`" for y in year_columns])}) as (year, value)"]

unpivoted = gu_like_wdi_filtered.select('Country Name', 'Indicator Name', *[col(y) for y in year_columns]) \
    .selectExpr('`Country Name`', '`Indicator Name`', *unpivot_exprs)

window_spec = Window.partitionBy('Country Name', 'Indicator Name').orderBy('year')

# Forward fill
filled = unpivoted.withColumn('value_ffill', last('value', ignorenulls=True).over(window_spec))

# Backward fill
window_spec_desc = Window.partitionBy('Country Name', 'Indicator Name').orderBy(col('year').desc())

filled = filled.withColumn('value_bfill', first('value_ffill', ignorenulls=True).over(window_spec_desc))

final_filled = filled.withColumn(
    'value',
    F.coalesce(F.col('value_ffill'), F.col('value_bfill'))
).drop('value_ffill', 'value_bfill')

# Pivoting back
silver_growup_df = final_filled.groupBy('Country Name', 'Indicator Name').pivot('year').agg(first('value'))

# Adding 0 values for remaining missing years in binary categories (leading nulls indicate 0 years warhist, peaceyears, etc and may be associated with newly recognized countries)
for col in year_range:
    silver_growup_df = silver_growup_df.withColumn(
        col, 
        F.coalesce(F.col(col), F.lit(0))
    )

print(f"Imputed dataset shape: {silver_growup_df.count()} rows x {len(silver_growup_df.columns)} columns")

null_counts = silver_growup_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in silver_growup_df.columns
]).collect()[0]

total_missing = sum(null_counts.asDict().values())

print(f"Missing values: {total_missing}")

# Checking where remaining missing values occur
silver_growup_df.show(10)

silver_growup_df.createOrReplaceTempView("silver_growup_imputed")

In [0]:
# Merging with WDI dataset
select_cols = ['Country Name', 'Indicator Name'] + year_range

combined_df = silver_wdi_df.select(select_cols).union(silver_growup_df.select(select_cols))

# Sorting by Country Name and Indicator Name
combined_df = combined_df.orderBy('Country Name', 'Indicator Name')

print(f"\nCombined dataset shape: {combined_df.count()} rows x {len(combined_df.columns)} columns")

null_counts = combined_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in combined_df.columns
]).collect()[0]

total_missing = sum(null_counts.asDict().values())

print(f"Missing values: {total_missing}")

print(f"Final imputed dataset shape: {combined_df.count()} rows x {len(combined_df.columns)} columns")

print("\nCombined dataset after imputation:")
combined_df.show(10)

# Saving GROWup data to table
silver_growup_df.write.option("delta.columnMapping.mode", "name").format("delta").mode("overwrite").saveAsTable("silver_growup_df")

# Saving merged data to table
combined_df.write.option("delta.columnMapping.mode", "name").format("delta").mode("overwrite").saveAsTable("silver_merged_df")

In [0]:
from pyspark.sql.functions import col, last, first, when, lit, avg, sum as _sum
 
# Tertiary dataset - Varieties of Democracy 
silver_vdem_df = bronze_vdem_df.drop('country_text_id', 'country_id', 'historical_date', 'project', 'historical', 'histname', 'codingstart', 'codingend', 'codingstart_contemp', 'codingend_contemp', 'codingstart_hist', 'codingend_hist', 'gap_index', 'COWcode'
)

# Subset of democracy indicators we wish to use (dataset contains hundreds of indicators)
keep_cols = ['country_name', 'year', 'v2x_api', 'v2x_api_codelow', 'v2x_api_codehigh', 'v2x_api_sd', 'v2x_mpi', 'v2x_mpi_codelow', 'v2x_mpi_codehigh', 'v2x_mpi_sd', 'v2x_freexp_altinf', 'v2x_freexp_altinf_codelow', 'v2x_freexp_altinf_codehigh', 'v2x_freexp_altinf_sd', 'v2x_frassoc_thick', 'v2x_frassoc_thick_codelow', 'v2x_frassoc_thick_codehigh', 'v2x_frassoc_thick_sd', 'v2x_suffr', 'v2xel_frefair', 'v2xel_frefair_codelow', 'v2xel_frefair_codehigh', 'v2xel_frefair_sd', 'v2x_elecoff', 'v2x_liberal', 'v2x_liberal_codelow', 'v2x_liberal_codehigh', 'v2x_liberal_sd', 'v2xcl_rol', 'v2xcl_rol_codelow', 'v2xcl_rol_codehigh', 'v2xcl_rol_sd', 'v2x_jucon', 'v2x_jucon_codelow', 'v2x_jucon_codehigh', 'v2x_jucon_sd', 'v2xlg_legcon', 'v2xlg_legcon_codelow', 'v2xlg_legcon_codehigh', 'v2xlg_legcon_sd', 'v2x_partip', 'v2x_partip_codelow', 'v2x_partip_codehigh', 'v2x_partip_sd', 'v2x_cspart', 'v2x_cspart_codelow', 'v2x_cspart_codehigh', 'v2x_cspart_sd', 'v2xdd_dd', 'v2xel_locelec', 'v2xel_locelec_codelow', 'v2xel_locelec_codehigh', 'v2xel_locelec_sd', 'v2xel_regelec', 'v2xel_regelec_codelow', 'v2xel_regelec_codehigh', 'v2xel_regelec_sd', 'v2x_polyarchy', 'v2x_polyarchy_codelow', 'v2x_polyarchy_codehigh', 'v2x_polyarchy_sd', 'v2x_libdem', 'v2x_libdem_codelow', 'v2x_libdem_codehigh', 'v2x_libdem_sd', 'v2x_partipdem', 'v2x_partipdem_codelow', 'v2x_partipdem_codehigh', 'v2x_partipdem_sd']

silver_vdem_df = silver_vdem_df.select(keep_cols)

# Filter first
silver_vdem_df = silver_vdem_df.filter(col('year').between(1960, 2020))

null_counts = silver_vdem_df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) 
    for c in silver_vdem_df.columns
]).collect()[0]

cols_with_nulls = [c for c, count in null_counts.asDict().items() if count > 0]

silver_vdem_df = silver_vdem_df.drop(*cols_with_nulls)

for field in silver_vdem_df.schema.fields:
    if isinstance(field.dataType, (IntegerType, LongType)) and field.name != 'year':
        silver_vdem_df = silver_vdem_df.withColumn(field.name, F.col(field.name).cast("double"))


In [0]:
# Converting VDEM dataset to WDI-like time series format
silver_vdem_df.createOrReplaceTempView("vdem_raw")
metadata_cols = ['country_name', 'year']
indicator_cols = [col for col in silver_vdem_df.columns if col not in metadata_cols]

# Unpivoting using stack
stack_expr = f"stack({len(indicator_cols)}, " + \
             ", ".join([f"'{ind}', `{ind}`" for ind in indicator_cols]) + \
             ") as (`Indicator Name`, value)"

vdem_long = silver_vdem_df.selectExpr(
    'country_name',
    'year',
    stack_expr
)

print(f"Melted dataframe shape: {vdem_long.count()} rows x {len(vdem_long.columns)} columns")
print("\nFirst few rows:")
vdem_long.show(5)
print(f"\nUnique indicators: {vdem_long.select('Indicator Name').distinct().count()}")

vdem_long.createOrReplaceTempView("vdem_long")

#Pivoting to get years as columns (WDI-like structure)
vdem_like_wdi = vdem_long.groupBy('country_name', 'Indicator Name') \
    .pivot('year') \
    .agg(first('value')) \

vdem_like_wdi = vdem_like_wdi.withColumnRenamed('country_name', 'Country Name') 

# Renaming columns to string type for consistency
for col_name in vdem_like_wdi.columns:
    if col_name not in ['Country Name', 'Indicator Name']:
        vdem_like_wdi = vdem_like_wdi.withColumnRenamed(col_name, str(col_name))

print("\nReshaped VDEM dataframe - Structure like WDI:")
print(f"Shape: {vdem_like_wdi.count()} rows x {len(vdem_like_wdi.columns)} columns")
print("\nColumn names (first 10):")
print(vdem_like_wdi.columns[:10])

vdem_like_wdi.createOrReplaceTempView("vdem_wdi_style")

In [0]:
# Missing value handling
year_columns = [col for col in vdem_like_wdi.columns 
                if col not in ['Country Name', 'Indicator Name']]

# Imputing remaining missing values with same forward/backward fill method as WDI (as not all indicator, country pairs have data for every year after converting years to cols)
unpivot_exprs = [f"stack({len(year_columns)}, {', '.join([f"'{y}', `{y}`" for y in year_columns])}) as (year, value)"]

unpivoted = vdem_like_wdi.select('Country Name', 'Indicator Name', *[col(y) for y in year_columns]) \
    .selectExpr('`Country Name`', '`Indicator Name`', *unpivot_exprs)

window_spec = Window.partitionBy('Country Name', 'Indicator Name').orderBy('year')

# Forward fill
filled = unpivoted.withColumn('value_ffill', last('value', ignorenulls=True).over(window_spec))

# Backward fill
window_spec_desc = Window.partitionBy('Country Name', 'Indicator Name').orderBy(col('year').desc())

filled = filled.withColumn('value_bfill', first('value_ffill', ignorenulls=True).over(window_spec_desc))

final_filled = filled.withColumn(
    'value',
    F.coalesce(F.col('value_ffill'), F.col('value_bfill'))
).drop('value_ffill', 'value_bfill')

# Pivoting back
silver_vdem_df = final_filled.groupBy('Country Name', 'Indicator Name').pivot('year').agg(first('value'))

# print("Imputation complete!")
print(f"Imputed dataset shape: {silver_vdem_df.count()} rows x {len(silver_vdem_df.columns)} columns")

null_counts = silver_vdem_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in silver_vdem_df.columns
]).collect()[0]

total_missing = sum(null_counts.asDict().values())

print(f"Missing values: {total_missing}")

silver_vdem_df.show(10)

silver_vdem_df.createOrReplaceTempView("silver_vdem_imputed")

In [0]:
# Merging VDEM dataset with previous merged dataset 
combined_df = spark.read.table("workspace.default.silver_merged_df")

vdem_combined_df = combined_df.unionByName(silver_vdem_df)

# Sorting by Country Name and Indicator Name
vdem_combined_df = vdem_combined_df.orderBy('Country Name', 'Indicator Name')

print(f"\nCombined VDEM dataset shape: {vdem_combined_df.count()} rows x {len(vdem_combined_df.columns)} columns")

null_counts = vdem_combined_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in vdem_combined_df.columns
]).collect()[0]

total_missing = sum(null_counts.asDict().values())

print(f"Missing VDEM values: {total_missing}")

# Saving VDEM data to table
silver_vdem_df.write.option("delta.columnMapping.mode", "name").format("delta").mode("overwrite").saveAsTable("silver_vdem_df")

# Saving merged data to table
vdem_combined_df.write.option("delta.columnMapping.mode", "name").format("delta").mode("overwrite").saveAsTable("silver_merged_vdem_df")

In [0]:
# Double-check nulls immediately before union
print("=== BEFORE UNION ===")

# Get exact row counts
combined_count = combined_df.count()
vdem_count = silver_vdem_df.count()
print(f"combined_df rows: {combined_count}")
print(f"silver_vdem_df rows: {vdem_count}")

# Get null counts for first few columns as a sample
sample_cols = ['Country Name', 'Indicator Name', '1960', '1961', '1962']

for col in sample_cols:
    nulls1 = combined_df.filter(F.col(col).isNull()).count()
    nulls2 = silver_vdem_df.filter(F.col(col).isNull()).count()
    if nulls1 > 0 or nulls2 > 0:
        print(f"{col} - combined_df nulls: {nulls1}, silver_vdem_df nulls: {nulls2}")